# Pilot: does the Grassmann geometry matter?

A small, quick version of the main experiment, run **before** the full protocol to see whether there is anything to find.

- **Models (5):** `grassmann` (the paper's Plücker features) against `sym`, `outer` and `concat` (same layer, same ~496 features, different pair feature), plus `attention` as the reference. `concat32` (concat at the paper's r = 32, 64 features) was added after the first pilot.
- **2 seeds × 5 epochs**, one learning rate (1e-3) for all. The paper also used one shared LR.
- **Validation only.** The test set is never evaluated (`--no-test`), so the full confirmatory run in `PROTOCOL.md` stays clean. This pilot is exploratory; it is not used for claims.

Time: ~40 min on an A100 (measured), ~2.5 h on an L4. If you upload the previous `results.zip` in the restore cell, only the new runs train (`concat32`: a few minutes on an A100).

Results are saved in `/content/results` (no Google Drive). Re-running a cell skips finished runs. Download them with the last cell before closing the runtime.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
import os
REPO, DATA, OUTDIR = '/content/grassman', '/content/data', '/content/results'
RUNS = f'{OUTDIR}/pilot.jsonl'
os.makedirs(OUTDIR, exist_ok=True)
if os.path.exists(REPO):
    !git -C {REPO} pull -q
else:
    !git clone -q https://github.com/thebnbrkr/grassman.git {REPO}
%cd {REPO}
!git log --oneline -1
!pip install -q datasets transformers

Only if resuming: set `RESTORE = True` to upload a previous `results.zip` so finished runs are skipped. On a first run leave it `False`.

In [ ]:
RESTORE = False
if RESTORE:
    from google.colab import files
    for name in files.upload():
        !cd /content && unzip -o -q "{name}"
!ls -la {OUTDIR}

Unit tests (about a minute). Stop if anything fails.

In [ ]:
!python -m pytest -q

In [ ]:
!python -m grassman.data --data-dir {DATA}

## Run the pilot

In [ ]:
import json, subprocess, sys

MIXERS = ['grassmann', 'sym', 'outer', 'concat', 'concat32', 'attention']
SEEDS  = [0, 1]
EPOCHS = 5
LR     = 1e-3

for seed in SEEDS:                       # seed-major: an interruption still leaves complete seeds
    for m in MIXERS:
        cmd = [sys.executable, '-m', 'grassman.train', '--mixer', m, '--seed', str(seed), '--lr', str(LR),
               '--epochs', str(EPOCHS), '--tag', 'pilot', '--data-dir', DATA, '--out', RUNS,
               '--skip-existing', '--no-test']
        print('>>', m, 'seed', seed, flush=True)
        subprocess.run(cmd, cwd=REPO, check=True)

## Results

Validation perplexity (best epoch) per model and seed, and the difference from `grassmann` within each seed. **Positive = the paper's Plücker features are better.**

How to read it: differences similar to the seed-to-seed spread are noise. For scale, the paper's two identical transformer runs differed by 2.1% (248.4 vs 253.6). A real effect should be larger than that and have the same sign in both seeds.

In [ ]:
from collections import defaultdict
rows = [json.loads(l) for l in open(RUNS)]
val = defaultdict(dict)
for r in rows:
    val[r['config']['mixer']][r['seed']] = r['best_val_ppl']
seeds = sorted({s for v in val.values() for s in v})

print(f"{'mixer':<10}" + ''.join(f"{'seed '+str(s):>12}" for s in seeds) + f"{'mean':>10}{'min':>10}")
for m in MIXERS:
    if m not in val: continue
    xs = [val[m][s] for s in seeds if s in val[m]]
    print(f"{m:<10}" + ''.join(f"{val[m].get(s, float('nan')):12.1f}" for s in seeds)
          + f"{sum(xs)/len(xs):10.1f}{min(xs):10.1f}")

print(f"\nother − grassmann (val ppl; positive = grassmann better)")
for m in MIXERS:
    if m == 'grassmann' or m not in val: continue
    d = [val[m][s] - val['grassmann'][s] for s in seeds if s in val[m] and s in val['grassmann']]
    pct = [100 * x / val['grassmann'][s] for x, s in zip(d, seeds)]
    print(f"{m:<10}" + ''.join(f"{x:+10.1f} ({p:+.1f}%)" for x, p in zip(d, pct)))

for m in MIXERS:
    if m in val and len(val[m]) > 1:
        a, b = list(val[m].values())[:2]
        print(f"seed spread {m:<10} {abs(a - b):6.1f} ({100 * abs(a - b) / min(a, b):.1f}%)")

In [ ]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(7, 4))
for r in rows:
    h = r['history']
    ax.plot([e['epoch'] for e in h], [e['val_ppl'] for e in h],
            label=f"{r['config']['mixer']} s{r['seed']}", ls='-' if r['seed'] == 0 else '--')
ax.set_xlabel('epoch'); ax.set_ylabel('validation perplexity'); ax.grid(alpha=.3); ax.legend(fontsize=8, ncol=2)
plt.savefig(f'{OUTDIR}/pilot_val_curves.png', dpi=150, bbox_inches='tight'); plt.show()

Download the results before the runtime is deleted.

In [ ]:
!cd /content && zip -qr results.zip results
from google.colab import files
files.download('/content/results.zip')